In [2]:
# setup
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Proyek_CRM_KELOMPOK')
sys.path.append(str(PROJECT_DIR))

from project_config import PROCESSED_DIR, OUTPUT_DIR, REPORT_DIR, FINAL_DATA_PATH

import pandas as pd

Mounted at /content/drive


In [4]:
#load risk scoring
df = pd.read_csv(PROCESSED_DIR / 'customer_risk_scoring.csv')

print(df.shape)
df.head()

(286, 58)


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,...,Politeness,Freshness,Temperature,Good Taste,Good Quantity,Output,Reviews,churn_risk,risk_score,risk_segment
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,...,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Moderately Important,Yes,Nil,0,0.108866,Low Risk
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,...,Very Important,Very Important,Very Important,Very Important,Very Important,Yes,Nil,0,0.198212,Low Risk
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,...,Very Important,Very Important,Important,Very Important,Moderately Important,Yes,"Many a times payment gateways are an issue, so...",0,0.140957,Low Risk
3,22,Female,Single,Student,No Income,Graduate,6,12.9473,77.5616,560019,...,Very Important,Very Important,Very Important,Very Important,Important,Yes,nil,0,0.163475,Low Risk
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,12.9850,77.5533,560010,...,Important,Important,Important,Very Important,Very Important,Yes,NIL,0,0.134960,Low Risk


In [5]:
# insert data nama
male_first_names = [
    'Andi', 'Budi', 'Fajar', 'Indra', 'Joko', 'Rama', 'Tono', 'Wahyu',
    'Zaki', 'Rizky', 'Bayu', 'Rafi', 'Agus', 'Dimas', 'Arif', 'Hendra',
    'Bagas', 'Yoga', 'Farhan', 'Doni'
]

female_first_names = [
    'Citra', 'Dewi', 'Eka', 'Gita', 'Hana', 'Kartika', 'Lina', 'Maya',
    'Nanda', 'Putri', 'Sari', 'Vina', 'Yuni', 'Dina', 'Nisa', 'Tari',
    'Ayu', 'Rani', 'Fitri', 'Nadia'
]

neutral_first_names = [
    'Alex', 'Raka', 'Dian', 'Ari', 'Sasha', 'Rizal', 'Tirta', 'Nara'
]

last_names = [
    'Pratama', 'Saputra', 'Wijaya', 'Lestari', 'Permana', 'Santoso',
    'Kurniawan', 'Ramadhani', 'Maulana', 'Pertiwi', 'Nugroho',
    'Wibowo', 'Hidayat', 'Putra', 'Salsabila', 'Firmansyah',
    'Ananda', 'Mahendra', 'Utami', 'Syahputra'
]

def make_dummy_name_by_gender(row, i):
    gender = str(row.get('Gender', '')).strip().lower()

    if gender == 'male':
        first = male_first_names[i % len(male_first_names)]
    elif gender == 'female':
        first = female_first_names[i % len(female_first_names)]
    else:
        first = neutral_first_names[i % len(neutral_first_names)]

    last = last_names[(i // 3) % len(last_names)]
    return f'{first} {last}'

df = df.reset_index(drop=True)

df['customer_id'] = [
    f'CUST-{str(i + 1).zfill(4)}'
    for i in range(len(df))
]

df['customer_name'] = [
    make_dummy_name_by_gender(row, i)
    for i, row in df.iterrows()
]

df[['customer_id', 'customer_name', 'Gender']].head(20)

,customer_id,customer_name,Gender
0,CUST-0001,Citra Pratama,Female
1,CUST-0002,Dewi Pratama,Female
2,CUST-0003,Fajar Pratama,Male
3,CUST-0004,Gita Saputra,Female
4,CUST-0005,Joko Saputra,Male
5,CUST-0006,Kartika Saputra,Female
6,CUST-0007,Tono Wijaya,Male
7,CUST-0008,Maya Wijaya,Female
8,CUST-0009,Nanda Wijaya,Female
9,CUST-0010,Putri Lestari,Female


In [6]:
df[['customer_name', 'Gender']].sample(20, random_state=42)

,customer_name,Gender
9,Putri Lestari,Female
267,Maya Pertiwi,Female
143,Indra Ramadhani,Male
212,Agus Nugroho,Male
227,Wahyu Firmansyah,Male
155,Hendra Wibowo,Male
283,Indra Salsabila,Male
73,Dimas Permana,Male
196,Bagas Santoso,Male
33,Dina Wibowo,Female


In [7]:
df['Gender'].value_counts()

,count
Gender,
Male,164
Female,122


In [14]:
#Fungsi ambil nilai
def get_value(row, col):
    if col in row.index:
        return str(row[col]).lower()
    return ''

In [15]:
# Buat reason code
def assign_reason_code(row):
    time_saving = get_value(row, 'Time saving')
    wait_time = get_value(row, 'Maximum wait time')
    ease = get_value(row, 'Ease and convenient')
    promo = get_value(row, 'More Offers and Discount')
    late_delivery = get_value(row, 'Late Delivery')
    food_quality = get_value(row, 'Good Food quality')

    if 'disagree' in time_saving or '15' in wait_time:
        return 'Time Risk'

    if 'disagree' in ease:
        return 'Convenience Risk'

    if 'disagree' in promo:
        return 'Promo Risk'

    if 'agree' in late_delivery:
        return 'Service Risk'

    if 'disagree' in food_quality:
        return 'Food Quality Risk'

    return 'General Risk'

df['reason_code'] = df.apply(assign_reason_code, axis=1)

df[['risk_score', 'risk_segment', 'reason_code']].head()

,risk_score,risk_segment,reason_code
0,0.108866,Low Risk,General Risk
1,0.198212,Low Risk,Service Risk
2,0.140957,Low Risk,Food Quality Risk
3,0.163475,Low Risk,General Risk
4,0.134960,Low Risk,Service Risk


In [16]:
# Buat CRM action
def assign_crm_action(row):
    segment = row['risk_segment']
    reason = row['reason_code']

    if segment == 'High Risk':
        if reason == 'Time Risk':
            return 'Fast delivery voucher dan rekomendasi restoran terdekat'
        if reason == 'Convenience Risk':
            return 'Panduan reorder cepat dan rekomendasi otomatis'
        if reason == 'Promo Risk':
            return 'Voucher personal dan campaign We Miss You'
        if reason == 'Service Risk':
            return 'Kompensasi delay dan prioritas merchant rating tinggi'
        if reason == 'Food Quality Risk':
            return 'Rekomendasi merchant kualitas tinggi dan voucher recovery'
        return 'Campaign retensi personal untuk mendorong pembelian ulang'

    if segment == 'Medium Risk':
        if reason == 'Time Risk':
            return 'Promo makan siang cepat dan reminder jam makan'
        if reason == 'Promo Risk':
            return 'Loyalty point dan bundle promo'
        return 'Promo personal dan rekomendasi restoran sesuai preferensi'

    return 'Referral reward, membership tier, dan reward review positif'

df['crm_action'] = df.apply(assign_crm_action, axis=1)

In [17]:
# Buat channel CRM
def assign_channel(row):
    if row['risk_segment'] == 'High Risk':
        return 'Push notification, WhatsApp, in-app'
    if row['risk_segment'] == 'Medium Risk':
        return 'Email, in-app, push notification'
    return 'In-app, email, referral link'

df['crm_channel'] = df.apply(assign_channel, axis=1)

In [18]:
#Buat KPI CRM
def assign_kpi(row):
    if row['risk_segment'] == 'High Risk':
        return 'Reorder rate, voucher redemption, penurunan churn risk'
    if row['risk_segment'] == 'Medium Risk':
        return 'Repeat order, frequency, campaign conversion'
    return 'Referral rate, retention rate, review rate'

df['crm_kpi'] = df.apply(assign_kpi, axis=1)

In [19]:
# Simpan dataset final
df.to_csv(FINAL_DATA_PATH, index=False)

print('Dataset final disimpan:', FINAL_DATA_PATH)

Dataset final disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/data/processed/food_delivery_crm_final_result.csv


In [21]:
# customer priority
priority_columns = [
    'customer_id',
    'customer_name',
    'risk_score',
    'risk_segment',
    'reason_code',
    'crm_action',
    'crm_channel',
    'crm_kpi',
    'Age',
    'Gender',
    'Marital Status',
    'Occupation',
    'Monthly Income',
    'Family size'
]

available_priority_columns = [col for col in priority_columns if col in df.columns]

customer_priority = (
    df[available_priority_columns]
    .sort_values('risk_score', ascending=False)
)

customer_priority.head(20)

,customer_id,customer_name,risk_score,risk_segment,reason_code,crm_action,crm_channel,crm_kpi,Age,Gender,Marital Status,Occupation,Monthly Income,Family size
249,CUST-0250,Rizky Lestari,0.881806,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",26,Male,Married,Employee,10001 to 25000,3
242,CUST-0243,Eka Pratama,0.880275,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",28,Female,Married,Employee,25001 to 50000,6
269,CUST-0270,Putri Pertiwi,0.879503,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",28,Female,Married,Employee,25001 to 50000,5
272,CUST-0273,Yuni Nugroho,0.874967,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",27,Female,Married,Self Employeed,More than 50000,6
266,CUST-0267,Lina Maulana,0.874857,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",26,Female,Married,Employee,10001 to 25000,2
218,CUST-0219,Farhan Hidayat,0.867651,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",29,Male,Single,Employee,25001 to 50000,3
229,CUST-0230,Putri Ananda,0.867533,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",26,Female,Married,Employee,Below Rs.10000,6
105,CUST-0106,Rama Firmansyah,0.865858,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",24,Male,Prefer not to say,Self Employeed,More than 50000,2
281,CUST-0282,Dewi Putra,0.851836,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",25,Female,Single,Employee,More than 50000,6
280,CUST-0281,Andi Putra,0.850032,High Risk,Time Risk,Fast delivery voucher dan rekomendasi restoran...,"Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch...",28,Male,Single,Employee,25001 to 50000,2


In [22]:
# Simpan customer priority table
priority_path = OUTPUT_DIR / 'customer_priority_table.csv'
customer_priority.to_csv(priority_path, index=False)

print('Customer priority table disimpan:', priority_path)

Customer priority table disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/customer_priority_table.csv


In [23]:
# Ringkasan reason code
reason_summary = (
    df.groupby(['risk_segment', 'reason_code'])
    .agg(
        total_customer=('reason_code', 'count'),
        avg_risk_score=('risk_score', 'mean')
    )
    .reset_index()
    .sort_values(['risk_segment', 'total_customer'], ascending=[True, False])
)

reason_summary

,risk_segment,reason_code,total_customer,avg_risk_score
0,High Risk,Time Risk,40,0.807369
5,Low Risk,Service Risk,139,0.198926
3,Low Risk,General Risk,39,0.169861
6,Low Risk,Time Risk,16,0.291902
4,Low Risk,Promo Risk,14,0.228297
2,Low Risk,Food Quality Risk,4,0.196384
1,Low Risk,Convenience Risk,1,0.374101
11,Medium Risk,Time Risk,14,0.593997
10,Medium Risk,Service Risk,9,0.526695
9,Medium Risk,Promo Risk,5,0.457113


In [24]:
#Simpan reason code summary
reason_path = OUTPUT_DIR / 'reason_code_summary.csv'
reason_summary.to_csv(reason_path, index=False)

print('Reason code summary disimpan:', reason_path)

Reason code summary disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/reason_code_summary.csv


In [25]:
#CRM strategy matrix
crm_strategy = pd.DataFrame({
    'segment': ['High Risk', 'Medium Risk', 'Low Risk'],
    'karakteristik': [
        'Skor risiko tinggi, sensitif pada waktu, kemudahan, atau value layanan',
        'Risiko sedang, masih bisa didorong dengan promo dan pengalaman konsisten',
        'Risiko rendah, pelanggan potensial untuk loyalitas dan referral'
    ],
    'strategi_utama': [
        'Fast delivery voucher, We Miss You campaign, kompensasi delay, rekomendasi restoran terdekat',
        'Loyalty point, bundle makan siang, rekomendasi restoran sesuai preferensi, promo personal',
        'Referral reward, membership tier, early access promo, reward review positif'
    ],
    'channel': [
        'Push notification, WhatsApp, in-app',
        'Email, in-app, push notification',
        'In-app, email, referral link'
    ],
    'kpi': [
        'Reorder rate, voucher redemption, penurunan churn risk',
        'Repeat order, frequency, campaign conversion',
        'Referral rate, retention rate, review rate'
    ]
})

crm_strategy

,segment,karakteristik,strategi_utama,channel,kpi
0,High Risk,"Skor risiko tinggi, sensitif pada waktu, kemud...","Fast delivery voucher, We Miss You campaign, k...","Push notification, WhatsApp, in-app","Reorder rate, voucher redemption, penurunan ch..."
1,Medium Risk,"Risiko sedang, masih bisa didorong dengan prom...","Loyalty point, bundle makan siang, rekomendasi...","Email, in-app, push notification","Repeat order, frequency, campaign conversion"
2,Low Risk,"Risiko rendah, pelanggan potensial untuk loyal...","Referral reward, membership tier, early access...","In-app, email, referral link","Referral rate, retention rate, review rate"


In [26]:
# Campaign Action Plan
campaign_action_plan = pd.DataFrame({
    'campaign_name': [
        'Fast Delivery Recovery',
        'We Miss You Voucher',
        'Lunch Time Retention',
        'Loyalty Point Booster',
        'Referral Loyal Customer'
    ],
    'target_segment': [
        'High Risk',
        'High Risk',
        'Medium Risk',
        'Medium Risk',
        'Low Risk'
    ],
    'target_reason_code': [
        'Time Risk',
        'Promo Risk',
        'Time Risk',
        'Promo Risk',
        'General Risk'
    ],
    'channel': [
        'Push notification',
        'WhatsApp dan in-app',
        'Push notification',
        'Email dan in-app',
        'Referral link'
    ],
    'offer': [
        'Voucher fast delivery untuk restoran terdekat',
        'Voucher khusus pembelian ulang',
        'Promo makan siang cepat',
        'Poin tambahan untuk 3 transaksi berikutnya',
        'Reward referral untuk pelanggan loyal'
    ],
    'kpi': [
        'Reorder rate',
        'Voucher redemption',
        'Lunch order conversion',
        'Repeat order frequency',
        'Referral rate'
    ]
})

campaign_action_plan

,campaign_name,target_segment,target_reason_code,channel,offer,kpi
0,Fast Delivery Recovery,High Risk,Time Risk,Push notification,Voucher fast delivery untuk restoran terdekat,Reorder rate
1,We Miss You Voucher,High Risk,Promo Risk,WhatsApp dan in-app,Voucher khusus pembelian ulang,Voucher redemption
2,Lunch Time Retention,Medium Risk,Time Risk,Push notification,Promo makan siang cepat,Lunch order conversion
3,Loyalty Point Booster,Medium Risk,Promo Risk,Email dan in-app,Poin tambahan untuk 3 transaksi berikutnya,Repeat order frequency
4,Referral Loyal Customer,Low Risk,General Risk,Referral link,Reward referral untuk pelanggan loyal,Referral rate


In [27]:
# Simpan campaign action plan
campaign_path = OUTPUT_DIR / 'campaign_action_plan.csv'
campaign_action_plan.to_csv(campaign_path, index=False)

print('Campaign action plan disimpan:', campaign_path)


Campaign action plan disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/campaign_action_plan.csv


In [28]:
# Simpan catatan CRAM
cram_notes = """
CRAM atau Customer Retention Action Matrix dipakai untuk mengubah hasil prediksi menjadi tindakan CRM.

Komponen CRAM:
1. Risk score
2. Risk segment
3. Reason code
4. CRM action
5. CRM channel
6. CRM KPI

Nilai utama CRAM:
Project tidak berhenti pada prediksi, tetapi menerjemahkan hasil model menjadi strategi retensi pelanggan.
"""

cram_notes_path = REPORT_DIR / 'cram_notes.txt'

with open(cram_notes_path, 'w') as f:
    f.write(cram_notes)

print('Catatan CRAM disimpan:', cram_notes_path)

Catatan CRAM disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/reports/cram_notes.txt


In [29]:
# Simpan CRM strategy matrix
crm_strategy_path = OUTPUT_DIR / 'crm_strategy_matrix.csv'
crm_strategy.to_csv(crm_strategy_path, index=False)

print('CRM strategy matrix disimpan:', crm_strategy_path)

CRM strategy matrix disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/outputs/crm_strategy_matrix.csv
